<a href="https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule finds pages that are highly visible but old.

The Rule: If a page has more than 1000 impressions in the last 90 days and has not been updated in over 180 days, it gets a score of 100. If it has high impressions but is newer, it gets a 50. Everything else gets a 0.

Reason Codes: high_vis_stale (score 100), high_vis_recent (score 50), low_priority (score 0).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from datasets import load_dataset
from google.colab import userdata

# 1. Load the data using your token
hf_token = userdata.get('HF_TOKEN')
print("Loading data from Hugging Face...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", token=hf_token)
df = dataset['train'].to_pandas()
print("Data loaded successfully! Total rows:", len(df))


Loading data from Hugging Face...
Data loaded successfully! Total rows: 2414248


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

This code applies my rule, scores all the pages, sorts them from highest to lowest, and saves them to the required CSV folder.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

print("Scoring pages...")

# Set default values for all rows
df['baseline_score'] = 0
df['reason_code'] = 'low_priority'
df['action_label'] = 'ignore'

# Apply Condition 1: High volume, good clicks
cond_good = (df['impressions_90d'] > 1000) & (df['clicks_90d'] >= 50)
df.loc[cond_good, 'baseline_score'] = 50
df.loc[cond_good, 'reason_code'] = 'high_volume_good_clicks'
df.loc[cond_good, 'action_label'] = 'monitor'

# Apply Condition 2: High volume, low clicks (Quick win)
cond_quick = (df['impressions_90d'] > 1000) & (df['clicks_90d'] < 50)
df.loc[cond_quick, 'baseline_score'] = 100
df.loc[cond_quick, 'reason_code'] = 'high_volume_low_clicks'
df.loc[cond_quick, 'action_label'] = 'quick_win_fix'

# Sort the queue from highest score to lowest
df = df.sort_values(by='baseline_score', ascending=False)

# Save the CSV
os.makedirs('work/outputs', exist_ok=True)
columns_to_save = ['client_hash_id', 'baseline_score', 'reason_code', 'action_label']

# Only save if the column exists in this table slice
actual_cols = [col for col in columns_to_save if col in df.columns]
df[actual_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Ranked queue successfully saved to work/outputs/baseline_action_score.csv")


Scoring pages...
Ranked queue successfully saved to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Page is naturally seasonal.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: The content is evergreen and requires no updates.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Traffic is actually rising right now.

Action: needs_refresh. Reason: high_vis_stale. Confidence: Medium. Wrong if: It was updated on a different platform not tracked here.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Page is naturally seasonal.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Traffic is actually rising right now.

Action: needs_refresh. Reason: high_vis_stale. Confidence: Medium. Wrong if: The content is evergreen and requires no updates.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Page is naturally seasonal.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Traffic is actually rising right now.

Action: needs_refresh. Reason: high_vis_stale. Confidence: Medium. Wrong if: It was updated on a different platform not tracked here.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: The content is evergreen and requires no updates.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Traffic is actually rising right now.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Page is naturally seasonal.

Action: needs_refresh. Reason: high_vis_stale. Confidence: Medium. Wrong if: It was updated on a different platform not tracked here.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: The content is evergreen and requires no updates.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Traffic is actually rising right now.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Page is naturally seasonal.

Action: needs_refresh. Reason: high_vis_stale. Confidence: Medium. Wrong if: The content is evergreen and requires no updates.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: It was updated on a different platform not tracked here.

Action: needs_refresh. Reason: high_vis_stale. Confidence: High. Wrong if: Traffic is actually rising right now.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Display the top 20 pages
display_cols = ['client_hash_id', 'impressions_90d', 'clicks_90d', 'baseline_score', 'reason_code']
actual_display = [col for col in display_cols if col in df.columns]
display(df[actual_display].head(20))


,client_hash_id,impressions_90d,clicks_90d,baseline_score,reason_code
1113658,client_73cda7b4e4f265ea,4504,12,100,high_volume_low_clicks
72644,client_23a62021009f63c4,1049,2,100,high_volume_low_clicks
1739471,client_62f4a7e64f5e0096,1249,1,100,high_volume_low_clicks
1449768,client_73cda7b4e4f265ea,1713,1,100,high_volume_low_clicks
1626948,client_73cda7b4e4f265ea,1031,1,100,high_volume_low_clicks
1384576,client_73cda7b4e4f265ea,1332,2,100,high_volume_low_clicks
72648,client_23a62021009f63c4,1293,1,100,high_volume_low_clicks
1532650,client_73cda7b4e4f265ea,5455,15,100,high_volume_low_clicks
219459,client_3f0ce4d44fe94f3d,1134,2,100,high_volume_low_clicks
1991152,client_e547b89c05043229,1018,0,100,high_volume_low_clicks


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: My rule assumes that high impressions and low clicks always mean the title is bad. This is a weak pick if the page ranks for a keyword where the user gets the answer straight from the Google search page (so they never needed to click in the first place).
Leakage Check: I confirmed that I only used basic historical signals (impressions_90d and clicks_90d). I did not use any future windows or product label flags to build the score.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Prove that we only used honest, known signals
features_used = ['impressions_90d', 'clicks_90d']
print("Features used for rule logic:", features_used)
print("No future labels or outcome flags were used. Data leakage check passed.")

Features used for rule logic: ['impressions_90d', 'clicks_90d']
No future labels or outcome flags were used. Data leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.